In [180]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# =========================
# 1) Load raw data
# =========================
climate_path = "../data/raw/climate/climate_data_from_1982.parquet"
barley_path = "../data/raw/yields/barley_yield_from_1982(in).csv"

climate_df = pd.read_parquet(climate_path)
barley_df = pd.read_csv(barley_path)
barley_df = barley_df.iloc[:, 0].str.split(";", expand=True)

print("Climate shape:", climate_df.shape)
print("Barley shape:", barley_df.shape)

# =========================
# 2) Clean barley data
# =========================
barley_df.columns = ["code_dep", "department", "year", "yield", "area", "production"]

# Strip spaces
barley_df["department"] = barley_df["department"].astype(str).str.strip()

# Convert numeric columns
for c in ["year", "yield", "area", "production"]:
    barley_df[c] = pd.to_numeric(barley_df[c], errors="coerce")

# Fill missing yield from production/area where possible
mask = barley_df["yield"].isna() & barley_df["production"].notna() & barley_df["area"].notna()
barley_df.loc[mask, "yield"] = barley_df.loc[mask, "production"] / barley_df.loc[mask, "area"]

# Drop rows still missing yield
barley_df = barley_df.dropna(subset=["yield"])

# Drop duplicates
barley_df = barley_df.drop_duplicates()

print(barley_df.info())
print(barley_df.head())
# Example: map code_dep -> department using barley table
climate_df = climate_df.merge(
    barley_df[["code_dep", "department"]].drop_duplicates(),
    on="code_dep",
    how="left"
)

# =========================
# 3) Prepare climate daily wide table (USE department, NOT code_dep)
# =========================
# Ensure department column exists in climate_df
# If your climate file still uses code_dep, map/rename it to department BEFORE this step.
# Here we assume climate_df has a 'department' column.

assert "department" in climate_df.columns, "climate_df must have a 'department' column"

climate_small = climate_df[["department", "year", "time", "metric", "value"]].copy()
climate_small["time"] = pd.to_datetime(climate_small["time"])

climate_wide = (
    climate_small
    .pivot_table(
        index=["department", "year", "time"],
        columns="metric",
        values="value",
        aggfunc="mean"
    )
    .reset_index()
)

# Convert precip from kg/m^2/s to mm/day
if "precipitation" in climate_wide.columns:
    climate_wide["precipitation"] = climate_wide["precipitation"] * 86400.0

print("Wide climate columns:", climate_wide.columns.tolist())
print(climate_wide.head())

# =========================
# 4) Helpers
# =========================
def max_consecutive_true(series):
    if series.isna().all() or len(series) == 0:
        return 0
    s = series.fillna(False).astype(int)
    groups = (s != s.shift()).cumsum()
    return int(s.groupby(groups).sum().max())

# =========================
# 5) Feature generator per (department, year)
# =========================
def climate_features_year(group):
    g = group.sort_values("time").copy()
    g["month"] = g["time"].dt.month

    feats = {}

    # Thresholds
    DRY_THRESH = 1.0          # mm/day
    HEAT_THRESH = 303.15      # 30°C in Kelvin
    FROST_THRESH = 273.15     # 0°C in Kelvin
    BASE_TEMP = 283.15        # 10°C in Kelvin for GDD

    # Required columns (names from your file)
    tmax = g["daily_maximum_near_surface_air_temperature"]
    tmean = g["near_surface_air_temperature"]
    precip = g["precipitation"]

    # -------- Annual features --------
    feats["tmean_mean"] = float(tmean.mean())
    feats["tmax_max"] = float(tmax.max())
    feats["tmax_p95"] = float(tmax.quantile(0.95))

    heat = tmax > HEAT_THRESH
    frost = tmean < FROST_THRESH
    dry = precip < DRY_THRESH
    wet = precip >= DRY_THRESH
    hot_dry = heat & dry

    feats["heat_days"] = int(heat.sum())
    feats["max_consecutive_heat_days"] = max_consecutive_true(heat)

    feats["frost_days"] = int(frost.sum())
    feats["max_consecutive_frost_days"] = max_consecutive_true(frost)

    feats["gdd"] = float((tmean - BASE_TEMP).clip(lower=0).sum())

    feats["total_precip"] = float(precip.sum())
    feats["rain_days"] = int((precip >= DRY_THRESH).sum())
    feats["max_1day_precip"] = float(precip.max())
    feats["p95_precip"] = float(precip.quantile(0.95))

    feats["max_consecutive_dry_days"] = max_consecutive_true(dry)
    feats["max_consecutive_wet_days"] = max_consecutive_true(wet)

    feats["max_3day_precip_sum"] = float(precip.rolling(3).sum().max())
    feats["max_5day_precip_sum"] = float(precip.rolling(5).sum().max())
    feats["max_7day_precip_sum"] = float(precip.rolling(7).sum().max())

    feats["precip_cv"] = float(precip.std() / precip.mean()) if precip.mean() != 0 else 0.0

    feats["hot_dry_days"] = int(hot_dry.sum())
    feats["max_consecutive_hot_dry_days"] = max_consecutive_true(hot_dry)

    # -------- Spring (Mar–May) --------
    spring = g[g["month"].isin([3, 4, 5])]
    if len(spring) > 0:
        tmax_s = spring["daily_maximum_near_surface_air_temperature"]
        tmean_s = spring["near_surface_air_temperature"]
        precip_s = spring["precipitation"]

        heat_s = tmax_s > HEAT_THRESH
        dry_s = precip_s < DRY_THRESH
        frost_s = tmean_s < FROST_THRESH

        feats["spring_heat_days"] = int(heat_s.sum())
        feats["spring_frost_days"] = int(frost_s.sum())
        feats["spring_max_consecutive_dry_days"] = max_consecutive_true(dry_s)
        feats["spring_gdd"] = float((tmean_s - BASE_TEMP).clip(lower=0).sum())
        feats["spring_total_precip"] = float(precip_s.sum())
    else:
        feats["spring_heat_days"] = 0
        feats["spring_frost_days"] = 0
        feats["spring_max_consecutive_dry_days"] = 0
        feats["spring_gdd"] = 0.0
        feats["spring_total_precip"] = 0.0

    # -------- Summer (Jun–Aug) --------
    summer = g[g["month"].isin([6, 7, 8])]
    if len(summer) > 0:
        tmax_u = summer["daily_maximum_near_surface_air_temperature"]
        tmean_u = summer["near_surface_air_temperature"]
        precip_u = summer["precipitation"]

        heat_u = tmax_u > HEAT_THRESH
        dry_u = precip_u < DRY_THRESH
        hot_dry_u = heat_u & dry_u

        feats["summer_heat_days"] = int(heat_u.sum())
        feats["summer_hot_dry_days"] = int(hot_dry_u.sum())
        feats["summer_max_consecutive_dry_days"] = max_consecutive_true(dry_u)
        feats["summer_gdd"] = float((tmean_u - BASE_TEMP).clip(lower=0).sum())
        feats["summer_total_precip"] = float(precip_u.sum())
        feats["summer_max_consecutive_heat_days"] = max_consecutive_true(heat_u)
    else:
        feats["summer_heat_days"] = 0
        feats["summer_hot_dry_days"] = 0
        feats["summer_max_consecutive_dry_days"] = 0
        feats["summer_gdd"] = 0.0
        feats["summer_total_precip"] = 0.0
        feats["summer_max_consecutive_heat_days"] = 0

    # -------- Winter (Dec–Feb) --------
    winter = g[g["month"].isin([12, 1, 2])]
    if len(winter) > 0:
        tmean_w = winter["near_surface_air_temperature"]
        frost_w = tmean_w < FROST_THRESH
        feats["winter_frost_days"] = int(frost_w.sum())
    else:
        feats["winter_frost_days"] = 0

    # -------- Monthly windows (Apr–Aug) --------
    for m in [4, 5, 6, 7, 8]:
        gm = g[g["month"] == m]
        if len(gm) > 0:
            tmax_m = gm["daily_maximum_near_surface_air_temperature"]
            tmean_m = gm["near_surface_air_temperature"]
            precip_m = gm["precipitation"]

            heat_m = tmax_m > HEAT_THRESH
            dry_m = precip_m < DRY_THRESH

            feats[f"m{m}_heat_days"] = int(heat_m.sum())
            feats[f"m{m}_max_consec_heat"] = max_consecutive_true(heat_m)
            feats[f"m{m}_dry_days"] = int(dry_m.sum())
            feats[f"m{m}_gdd"] = float((tmean_m - BASE_TEMP).clip(lower=0).sum())
            feats[f"m{m}_precip_sum"] = float(precip_m.sum())
        else:
            feats[f"m{m}_heat_days"] = 0
            feats[f"m{m}_max_consec_heat"] = 0
            feats[f"m{m}_dry_days"] = 0
            feats[f"m{m}_gdd"] = 0.0
            feats[f"m{m}_precip_sum"] = 0.0

    return pd.Series(feats)

# =========================
# 6) Build climate feature table (GROUP BY department, year)
# =========================
climate_features = (
    climate_wide
    .groupby(["department", "year"], group_keys=False)
    .apply(climate_features_year)
    .reset_index()
)

print("Climate features shape:", climate_features.shape)
print(climate_features.head())

# =========================
# 7) Merge with barley (ON department, year)
# =========================
df_merged = barley_df.merge(
    climate_features,
    on=["department", "year"],
    how="left"
)

print("Merged shape:", df_merged.shape)
print(df_merged.head())

# =========================
# 8) Fill remaining missing numeric values (except target)
# =========================
num_cols = df_merged.select_dtypes(include=np.number).columns
for col in num_cols:
    if col != "yield":
        df_merged[col] = df_merged[col].fillna(df_merged[col].median())

print(df_merged.info())
print(df_merged.isna().sum().sort_values(ascending=False).head())

# =========================
# 9) Simple linear detrend (optional, keep both)
# =========================
coef = np.polyfit(df_merged["year"], df_merged["yield"], 1)
trend = np.polyval(coef, df_merged["year"])
df_merged["yield_detrended"] = df_merged["yield"] - trend

# =========================
# 10) Save processed dataset
# =========================
out_path = "../data/processed/barley_climate_model_ready.csv"
df_merged.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Final shape:", df_merged.shape)


Climate shape: (13540116, 7)
Barley shape: (3583, 6)
<class 'pandas.core.frame.DataFrame'>
Index: 3461 entries, 0 to 3582
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   code_dep    3461 non-null   object 
 1   department  3461 non-null   object 
 2   year        3461 non-null   int64  
 3   yield       3461 non-null   float64
 4   area        3461 non-null   float64
 5   production  3458 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 189.3+ KB
None
  code_dep department  year     yield     area  production
0       82        Ain  1982  3.950080  16065.0     63458.0
1       83        Ain  1983  2.648276  14500.0     38400.0
2       84        Ain  1984  4.822580  15500.0     74750.0
3       85        Ain  1985  4.196770  15500.0     65050.0
4       86        Ain  1986  3.598450  12900.0     46420.0
Wide climate columns: ['department', 'year', 'time', 'daily_maximum_near_surface_air_temperatur

/var/folders/tc/ddy9q3l50gv9sy77lbp_p_cr0000gn/T/ipykernel_32382/712289422.py:237: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(climate_features_year)


Climate features shape: (69, 59)
  department  year  tmean_mean    tmax_max    tmax_p95  heat_days  max_consecutive_heat_days  frost_days  max_consecutive_frost_days          gdd  total_precip  rain_days  max_1day_precip  \
0        Ain  1982  284.088562  306.059875  301.214465       11.0                        7.0        11.0                         4.0  1229.868164   2292.475342      235.0        41.052830   
1        Ain  1983  283.052216  303.138611  299.784766        0.0                        0.0        14.0                         7.0  1019.569214   2666.141357      256.0        39.644939   
2        Ain  1984  283.335114  303.565155  299.290634        2.0                        1.0        23.0                         7.0  1015.322388   2303.400146      255.0        46.062279   
3        Ain  1985  283.212646  303.901123  300.743225        3.0                        2.0        25.0                         8.0  1117.411987   2557.704834      252.0        64.226509   
4        Ain

#